In [10]:
import sys
import re
sys.path.append("../src")

## 1. Load raw data

In [11]:
from data_loader import load_row
from preprocessing import substitute_template_vars, clean_description, drop_noise_columns, parse_dates
df = load_row("../data/customer_support_tickets.csv")

## 2. Before/after: template substitution

In [12]:
assert df["Ticket Description"].str.contains(r"\{product_purchased\}", regex=True).sum() != 0

In [13]:
sample = df["Ticket Description"].sample(3, random_state=42)
print("--- before ---")
print(sample.to_string())

df = substitute_template_vars(df)

print("\n--- after ---")
print(df.loc[sample.index, "Ticket Description"].to_string())

--- before ---
4830    I'm having an issue with the {product_purchase...
7075    I'm having trouble connecting my {product_purc...
4715    I'm having an issue with the {product_purchase...

--- after ---
4830    I'm having an issue with the Roomba Robot Vacu...
7075    I'm having trouble connecting my Roomba Robot ...
4715    I'm having an issue with the Philips Hue Light...


In [14]:
assert df["Ticket Description"].str.contains(r"\{product_purchased\}", regex=True).sum() == 0

## 3. Clean description

In [15]:
sample_idx = df["Ticket Description"].apply(
    lambda t: re.sub(r"\s+", " ", t).strip() != t
).idxmax()
print("--- before ---")
print(repr(df.loc[sample_idx, "Ticket Description"]))

df["Ticket Description"] = df["Ticket Description"].apply(clean_description)

print("\n--- after ---")
print(repr(df.loc[sample_idx, "Ticket Description"]))

--- before ---
"I'm having an issue with the GoPro Hero. Please assist.\n\nYour billing zip code is: 71701.\n\nWe appreciate that you have requested a website address.\n\nPlease double check your email address. I've tried troubleshooting steps mentioned in the user manual, but the issue persists."

--- after ---
"I'm having an issue with the GoPro Hero. Please assist. Your billing zip code is: 71701. We appreciate that you have requested a website address. Please double check your email address. I've tried troubleshooting steps mentioned in the user manual, but the issue persists."


## 4. Drop noise columns

In [16]:
df = drop_noise_columns(df)
assert not {"Resolution", "Customer Name", "Customer Email"} & set(df.columns)

## 5. Parse dates

In [17]:
df = parse_dates(df)
print(df.dtypes)   # First Response Time / Time to Resolution / Date of Purchase -> datetime64
df["response_time_hours"].describe()

Ticket ID                                int64
Customer Age                             int64
Customer Gender                            str
Product Purchased                          str
Date of Purchase                datetime64[us]
Ticket Type                                str
Ticket Subject                             str
Ticket Description                         str
Ticket Status                              str
Ticket Priority                            str
Ticket Channel                             str
First Response Time             datetime64[us]
Time to Resolution              datetime64[us]
Customer Satisfaction Rating           float64
response_time_hours                    float64
dtype: object


count    2769.000000
mean       -0.057704
std         9.564112
min       -23.233333
25%        -6.933333
50%         0.166667
75%         6.483333
max        23.466667
Name: response_time_hours, dtype: float64

## 6. Summary

**What we did:**
1. Replaced the template variable `{product_purchased}` with the real product name in all 8,469 descriptions
2. Removed extra spaces and line breaks in `Ticket Description` (this changed about 6,158 of 8,469 rows)
3. Dropped noisy columns: `Resolution` (Faker junk), `Customer Name`, `Customer Email`
4. Converted `Date of Purchase`, `First Response Time`, `Time to Resolution` to datetime, and calculated `response_time_hours`

**Finding:** `response_time_hours` is symmetric around 0 (from -23 to +23 hours, only available for about 33% of tickets). This confirms what we found in EDA — `First Response Time` and `Time to Resolution` are synthetic, randomly generated timestamps with no real causal link between them. We can't use this column as a real SLA metric, but it can still be a useful structural feature for customer segmentation (Phase 7).

**Final shape:** see the cell below.

**Next step:** Feature Engineering (Phase 4) — TF-IDF and SBERT embeddings for `Ticket Description`, UMAP for dimensionality reduction.

In [18]:
df.shape

(8469, 15)